In [ ]:
import cv2
import numpy as np

# Constants
LOWER_BLUE = np.array([100, 60, 60])
UPPER_BLUE = np.array([130, 255, 255])
LOWER_RED = np.array([0, 100, 100])
UPPER_RED = np.array([180, 410, 410])
Kp = 0.01

# Function to segment the frame
def segment_frame(frame_bgr):
    # Convert BGR to HSV
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    
    # Create masks for blue and red
    mask_blue = cv2.inRange(hsv, LOWER_BLUE, UPPER_BLUE)
    mask_red = cv2.inRange(hsv, LOWER_RED, UPPER_RED)
    
    # Find contours for blue and red
    contours_blue, _ = cv2.findContours(mask_blue, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_red, _ = cv2.findContours(mask_red, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    return mask_blue, mask_red, contours_blue, contours_red

# Function to classify the scene
def classify_scene(labels):
    # Placeholder for classification logic
    return "linea_recta"  # Default class if no specific criteria met

# Function to detect red mark type
def detect_red_mark(labels):
    # Placeholder for detection logic
    return "sin_marca"  # Default class if no specific criteria met

# Function to estimate line error
def estimate_line_error(labels):
    return 0.0  # Placeholder value, replace with actual calculation

# Function to compute control command
def compute_control(error, kp=Kp):
    steering = kp * error
    steering = np.clip(steering, -1, 1)
    return steering

# Function to draw overlays on the frame
def draw_overlay(frame, results):
    cv2.putText(frame, f"Scene: {results['scene']}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Mark: {results['mark']}", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Orientation: {results['orientation']}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Selected Exit: {results['selected_exit']}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Error: {results['error']:.2f}", (10, 190), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Steering: {results['steering']:.2f}", (10, 230), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

# Function to process the video
def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    while cap.isOpened():
        ret, frame_bgr = cap.read()
        if not ret:
            break

        mask_blue, mask_red, contours_blue, contours_red = segment_frame(frame_bgr)

        # Placeholder for scene classification
        labels = {"blue_contours": contours_blue, "red_contours": contours_red}
        results = {
            "scene": classify_scene(labels),
            "mark": detect_red_mark(labels),
            "orientation": "adelante",
            "selected_exit": "1",
            "error": estimate_line_error(labels),
            "steering": None
        }
        
        results["steering"] = results["error"]

        draw_overlay(frame_bgr, results)

        out.write(frame_bgr)
        cv2.imshow('Processed Frame', frame_bgr)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

# Main block
if __name__ == "__main__":
    input_video = "video1.mp4"
    output_video = "resultado_robot.mp4"
    process_video(input_video, output_video)

: 